In [18]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd 
from lightkurve import LightCurve
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve 
import lightkurve as lk

for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize',
            'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 15

search_result = search_lightcurve("TOI 2631") 
print(search_result)
lc2min = search_result[2].download() 
lc2 = lc2min.remove_nans().remove_outliers()
lc2 = lc2[lc2.quality == 0]
lc2_m = lc2.normalize().remove_nans()

x2min = np.ascontiguousarray(lc2_m.time.value, dtype=np.float64) 
y2min = np.ascontiguousarray(lc2_m.flux, dtype=np.float64)
yerr2min = np.ascontiguousarray(lc2_m.flux_err, dtype=np.float64)
lcquality = np.ascontiguousarray(lc2_m.quality, dtype=np.float64)

SearchResult containing 19 data products.

 #     mission     year       author      exptime target_name distance
                                             s                 arcsec 
--- -------------- ---- ----------------- ------- ----------- --------
  0 TESS Sector 28 2020              SPOC     120   140067837      0.0
  1 TESS Sector 68 2023              SPOC     120   140067837      0.0
  2 TESS Sector 95 2025              SPOC     120   140067837      0.0
  3 TESS Sector 01 2018         TESS-SPOC    1800   140067837      0.0
  4 TESS Sector 28 2020         TESS-SPOC     600   140067837      0.0
  5 TESS Sector 68 2023         TESS-SPOC     200   140067837      0.0
  6 TESS Sector 01 2018               QLP    1800   140067837      0.0
  7 TESS Sector 28 2020               QLP     600   140067837      0.0
  8 TESS Sector 68 2023               QLP     200   140067837      0.0
  9 TESS Sector 95 2025               QLP     200   140067837      0.0
 10 TESS Sector 01 2018 GSFC-ELEAN

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
from astropy.timeseries import BoxLeastSquares

%matplotlib qt
plt.rcParams['font.size'] = '16'

# ---------------- 0. Dados ----------------
t = np.asarray(x2min, float)
f = np.asarray(y2min, float)
ok = np.isfinite(t) & np.isfinite(f)
t, f = t[ok], f[ok]
s = np.argsort(t); t, f = t[s], f[s]
f = f / np.nanmedian(f)

# ---------------- 1. Catálogo: TOI 2631.01 (TIC 140067837) ----------------
TIC_LABEL = 'TIC 140067837'
P_CAT     = 6.6314357                    # d   (TOIs / TESS Project)
T0_CAT    = 2459085.301994 - 2457000.0   # BTJD (WASP+TESS, Hellier)
DEPTH_CAT = 10550.9e-6                   # ExoFOP
RP_RE     = 13.19

# R* implícito por Rp e profundidade (checagem de consistência)
RS_RSUN = RP_RE * 6371.0 / (695700.0 * np.sqrt(DEPTH_CAT))
DEPTH_EXP = DEPTH_CAT
DUR_EXP   = 3.5 / 24.0                   # d  (~3.5 h; 11.5 h do ExoFOP é implausível)


# ---------------- 2. Detrending (mediana movel) ----------------
FLATTEN, WIN = True, 0.75                # janela >> duracao do transito

def _msize(n_req, n_dat):
    s = min(int(n_req), int(n_dat))
    if s % 2 == 0: s -= 1
    return max(s, 3)

if FLATTEN:
    cad   = np.median(np.diff(t))
    nwin  = round(WIN / cad)
    gaps  = np.where(np.diff(t) > 0.3)[0] + 1
    trend = np.concatenate([median_filter(c, size=_msize(nwin, c.size), mode='nearest')
                            for c in np.split(f, gaps)])
    f = f / trend

# ---------------- 3. Clip de flares (nunca corta o transito) ----------------
keep = np.ones(t.size, bool)
med  = np.median(f[keep])
sig  = 1.4826 * np.median(np.abs(f[keep] - med))
lo   = med - max(8.0*sig, 2.0*DEPTH_EXP)
keep &= (f < med + 3.0*sig) & (f > lo)

tb, fb = t[keep], f[keep]

# ---------------- 4. BLS em torno de P_CAT ----------------
bls  = BoxLeastSquares(tb, fb)
per  = np.linspace(P_CAT - 0.30, P_CAT + 0.30, 30000)
durs = np.array([0.06, 0.08, 0.10, 0.13, 0.16, 0.20, 0.25])
res  = bls.power(per, durs, objective='snr')

i     = int(np.argmax(res.power))
P_BLS = res.period[i]
T0    = res.transit_time[i]
DUR   = res.duration[i]
DEP   = res.depth[i]

mad = 1.4826*np.median(np.abs(res.power - np.median(res.power)))
sde = (res.power[i] - np.median(res.power)) / mad
dphi = (T0 - T0_CAT + 0.5*P_CAT) % P_CAT - 0.5*P_CAT


stats = bls.compute_stats(P_BLS, DUR, T0)


# ---------------- 5. Periodograma BLS ----------------
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(res.period, res.power, 'k', lw=1)
ax.axvline(P_CAT, color='tab:red',  ls='--', lw=2, label=f'catalogo {P_CAT:.4f} d')
ax.axvline(P_BLS, color='tab:blue', ls=':',  lw=2, label=f'BLS {P_BLS:.4f} d')
ax.set_xlabel('Periodo [dias]', fontsize=16)
ax.set_ylabel('Potencia BLS (SNR)', fontsize=16)
ax.legend(fontsize=13)
plt.tight_layout()

# ---------------- 6. Curva de luz com transitos previstos ----------------
n0 = int(np.floor((t.min() - T0)/P_BLS)) - 1
n1 = int(np.ceil ((t.max() - T0)/P_BLS)) + 1
tc = T0 + P_BLS*np.arange(n0, n1+1)
tc = tc[(tc > t.min()) & (tc < t.max())]

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(t, f, 'k', lw=0.6, zorder=1, label=TIC_LABEL)
for k, c in enumerate(tc):
    ax.axvspan(c-DUR/2, c+DUR/2, color='tab:red', alpha=0.25, zorder=0,
               label='transito previsto' if k == 0 else None)
    ax.axvline(c, color='tab:red', ls='--', lw=1.2, zorder=2)
ax.set_xlabel('Tempo - 2457000 [BTJD dias]', fontsize=16)
ax.set_ylabel('Fluxo Normalizado', fontsize=16)
ax.legend(fontsize=13)
plt.xticks(fontsize=16); plt.yticks(fontsize=16)
plt.tight_layout()

# ---------------- 7. Fase dobrada + binagem ----------------
ph  = (tb - T0 + 0.5*P_BLS) % P_BLS - 0.5*P_BLS
sel = np.abs(ph) < 4*DUR

nb    = 120
edges = np.linspace(-4*DUR, 4*DUR, nb+1)
idx   = np.digitize(ph[sel], edges) - 1
xb = np.array([np.mean(ph[sel][idx == j]) if np.any(idx == j) else np.nan for j in range(nb)])
yb = np.array([np.mean(fb[sel][idx == j]) if np.any(idx == j) else np.nan for j in range(nb)])
eb = np.array([np.std(fb[sel][idx == j])/np.sqrt(np.sum(idx == j)) if np.sum(idx == j) > 1 else np.nan
               for j in range(nb)])

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(ph[sel]*24, fb[sel], '.', color='0.75', ms=2, zorder=1)
ax.errorbar(xb*24, yb, yerr=eb, fmt='o', color='k', ms=5, lw=1.2, zorder=2)
ax.axhline(1.0, color='0.4', lw=1)
ax.axhline(1.0 - DEPTH_EXP, color='tab:red', ls='--', lw=1.5,
           label=f'profundidade esperada ({DEPTH_EXP*1e6:.0f} ppm)')
ax.axvspan(-DUR/2*24, DUR/2*24, color='tab:blue', alpha=0.12)
ax.set_xlabel('Tempo desde o meio do transito [horas]', fontsize=16)
ax.set_ylabel('Fluxo Normalizado', fontsize=16)
ax.set_title(f'Fase dobrada: P = {P_BLS:.5f} d, T0 = {T0:.5f} BTJD', fontsize=17)
ax.legend(fontsize=13)
plt.xticks(fontsize=16); plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()